#Chapter 6 - Building Visual Hierarchies with CNNs
An image may look effortless to a human, but to a neural network it begins as a grid of numbers. The challenge is to turn that grid into structure, edges, contours, textures, parts, and eventually complete objects. Convolutional neural networks excel at this because they discover visual structure one local pattern at a time, gradually building a hierarchy of visual understanding.


#Listing 6-1: Using a pretrained YOLOv5 model to detect objects in an image
This example lets you upload an image and count the objects in it using a pretrained YOLOv5 object-detection model. YOLOv5, short for You Only Look Once, version 5, is a CNN-based detector from Ultralytics designed for real-time performance. This example clones the YOLOv5 repository and loads the model locally for transparency, but the same family of models can also be loaded through PyTorch Hub.

In [ ]:
# ------------------------------------------------------------------
# Step 1: Set up YOLOv5 and install dependencies
# ------------------------------------------------------------------
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -r requirements.txt

# ------------------------------------------------------------------
# Step 2: Imports
# ------------------------------------------------------------------
import torch
import cv2
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import io

# ------------------------------------------------------------------
# Step 3: Upload an image
# ------------------------------------------------------------------
uploaded = files.upload()
image_path = next(iter(uploaded))  # take the first uploaded file

# ------------------------------------------------------------------
# Step 4: Load pretrained model
#         This example uses the medium-sized YOLOv5m variant.
# ------------------------------------------------------------------
model = torch.hub.load('.', 'yolov5m', source='local')

# ------------------------------------------------------------------
# Step 5: Run inference
# ------------------------------------------------------------------
results = model(image_path)

# ------------------------------------------------------------------
# Step 6: Count detected classes
# ------------------------------------------------------------------
detections = results.pandas().xyxy[0]
class_names = detections['name'].tolist()
counts = Counter(class_names)

print("\nDetected Objects:")
for label, count in counts.items():
    print(f"- {label}: {count}")

# ------------------------------------------------------------------
# Step 7: Display image with bounding boxes
# ------------------------------------------------------------------
results.render()
img = Image.fromarray(results.ims[0])
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title("Detected Objects")
plt.show()

#Listing 6-2: Video object detection implemented as repeated image detection using YOLOv5
This example runs YOLOv5 on an uploaded video inside a Colab notebook and prints a compact per-second table. With additional components around the model, the same pattern can support traffic monitoring, wildlife observation, or automated inspection on a conveyor belt.

In [ ]:
# ----------------------------------------------------------------------
# Step 1: Suppress nonessential warnings (including AMP FutureWarnings)
# ----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------------------------------------------------
# Step 2: Setup and imports
# ----------------------------------------------------------------------
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -r requirements.txt

import torch
import cv2
from collections import Counter, defaultdict
from google.colab import files
import math

# ----------------------------------------------------------------------
# Step 3: Load pretrained model
# ----------------------------------------------------------------------
model = torch.hub.load('.', 'yolov5m', source='local')
model.conf = 0.25  # confidence threshold (optional)

# ----------------------------------------------------------------------
# Step 4: Upload video
# ----------------------------------------------------------------------
print("Please upload a video file (e.g., .mp4)")
uploaded = files.upload()
video_source = next(iter(uploaded.keys()))

cap = cv2.VideoCapture(video_source)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video source: {video_source}")

# ----------------------------------------------------------------------
# Step 5: Determine FPS (fallback if missing)
# ----------------------------------------------------------------------
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or math.isnan(fps) or fps < 1:
    fps = 30.0

# ----------------------------------------------------------------------
# Step 6: Limit runtime for notebook safety
# ----------------------------------------------------------------------
MAX_SECONDS = 15
max_frames = int(MAX_SECONDS * fps)

# ----------------------------------------------------------------------
# Step 7: Store per-second MAX counts (non-accumulating)
# ----------------------------------------------------------------------
per_second_max = defaultdict(Counter)

frame_idx = 0  # 0-based
while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if frame_idx >= max_frames:
        break

    second = int(frame_idx / fps)
    frame_idx += 1

    # OpenCV uses BGR; YOLO expects RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Run inference
    results = model(frame_rgb)
    detections = results.pandas().xyxy[0]
    frame_counts = Counter(detections["name"].tolist())

    # Keep the maximum count seen in any frame during this second
    for label, count in frame_counts.items():
        if count > per_second_max[second][label]:
            per_second_max[second][label] = count

cap.release()

# ----------------------------------------------------------------------
# Step 8: Final per-second table
# ----------------------------------------------------------------------
print("\nPer-second detection summary (max per-frame counts):\n")

for second in sorted(per_second_max.keys()):
    counts = per_second_max[second]
    summary = ", ".join(f"{k}={v}" for k, v in counts.most_common()) if counts else "(none)"
    print(f"t={second:2d}s | {summary}")


#Listing 6-3: Removing backgrounds with a segmentation CNN (transparent output)
This example uses a pretrained DeepLabV3 network. Like other convolutional models, DeepLabV3 begins by detecting local patterns such as edges, corners, and textures. Deeper layers combine those patterns into higher-level shapes. The key difference appears at the output stage.

In [ ]:
# ----------------------------------------------------------------------
# Step 1: Imports and Setup
# ----------------------------------------------------------------------
import io
import numpy as np
import torch
from torchvision import models
from PIL import Image
from google.colab import files
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------------------------------------------
# Step 2: Load pretrained DeepLabV3 model and its transforms
# ----------------------------------------------------------------------
weights = models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
model = models.segmentation.deeplabv3_resnet50(weights=weights).to(device).eval()
preprocess = weights.transforms()

# ----------------------------------------------------------------------
# Step 3: Upload an image
# ----------------------------------------------------------------------
print("Please upload a photo")
uploaded = files.upload()
filename = next(iter(uploaded.keys()))

img = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

# ----------------------------------------------------------------------
# Step 4: Run the image through the CNN
# ----------------------------------------------------------------------
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_tensor)["out"][0]  # [num_classes, H, W]

labels = output.argmax(0).cpu().numpy()

# ----------------------------------------------------------------------
# Step 5: Build a foreground mask: anything not background (class 0)
# ----------------------------------------------------------------------
foreground_mask = labels != 0

# ----------------------------------------------------------------------
# Step 6: Make the background transparent (RGBA + alpha channel)
# ----------------------------------------------------------------------
h, w = labels.shape
img_resized = img.resize((w, h)).convert("RGBA")
img_np = np.array(img_resized).copy()

alpha = np.where(foreground_mask, 255, 0).astype(np.uint8)
img_np[..., 3] = alpha

# ----------------------------------------------------------------------
# Step 7: Display the result
# ----------------------------------------------------------------------
result_pil = Image.fromarray(img_np)
print("Showing result…")
display(result_pil)

# ----------------------------------------------------------------------
# Step 8: Optional: save a PNG with transparency
# ----------------------------------------------------------------------
out_name = "background_removed.png"
result_pil.save(out_name)
print("Saved:", out_name)
